Load Data
```
# This is formatted as code
```



In [8]:
import io
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

from google.colab import drive
drive.mount('/content/drive')
TS_DIR = '/content/drive/MyDrive/ehydro/anchorage/timeseries_v1'

ts = np.load(f'{TS_DIR}/timeseries.npz')
dates = pd.to_datetime(ts["dates"])
windows = ts['windows']

print(len(dates), 'maps,', dates.min().date(), 'to', dates.max().date())
print(windows.shape, 'samples')
ts.files

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
697 maps, 2014-05-01 to 2026-10-15
(515, 5) samples


['maps',
 'dredged',
 'core',
 'windows',
 'masks',
 'split',
 'K',
 'STEP',
 'HORIZON',
 'survey_id',
 'dates',
 'row0',
 'col0',
 'cell']

Get the tides

In [ ]:
TIDE_URL = 'https://api.tidesandcurrents.noaa.gov/api/prod/datagetter'

years = []
for year in range(2014, 2027):
    params = {'station': '9455920', 'product': 'high_low', 'datum': 'MLLW', 'units': 'english',
              'time_zone': 'gmt', 'format': 'csv',
              'begin_date': f'{year}0101', 'end_date': f'{year}1231'}
    text = requests.get(TIDE_URL, params=params).text
    years.append(pd.read_csv(io.StringIO(text)))

tides = pd.concat(years, ignore_index=True)
tides.columns = tides.columns.str.strip()
tides = tides.rename(columns={'Date Time': 'time', 'Water Level': 'level_ft', 'TY': 'type'})
tides['time'] = pd.to_datetime(tides.time)
tides['type'] = tides.type.str.strip()
tides = tides[['time', 'level_ft', 'type']]
print(len(tides), 'tides,', tides.time.min().date(), 'to', tides.time.max().date())
tides.head(8)

17631 tides, 2014-01-01 to 2026-09-30


,time,level_ft,type
0,2014-01-01 02:54:00,31.772,HH
1,2014-01-01 10:30:00,-3.622,LL
2,2014-01-01 16:00:00,31.243,H
3,2014-01-01 22:24:00,3.514,L
4,2014-01-02 03:36:00,33.855,HH
5,2014-01-02 11:24:00,-3.642,LL
6,2014-01-02 16:48:00,31.857,H
7,2014-01-02 23:24:00,2.041,L


In [ ]:
day = tides.time.dt.floor("D")
daily = tides.groupby(day).level_ft.agg(['max', 'min'])  # one row per day: highest and lowest level
daily['range_ft'] = daily['max'] - daily['min']
daily.head(8)


,max,min,range_ft
time,,,
2014-01-01,31.772,-3.622,35.394
2014-01-02,33.855,-3.642,37.497
2014-01-03,33.091,-4.370,37.461
2014-01-04,32.211,-4.544,36.755
2014-01-05,31.995,-2.772,34.767
2014-01-06,31.470,-0.679,32.149
2014-01-07,30.440,0.833,29.607
2014-01-08,29.409,2.461,26.948


In [ ]:
tide_range_before = np.full(len(dates), np.nan)       # 697 empty slots, one per map

for p, d in enumerate(dates):
    d = d.normalize()                                 # survey date without the time of day
    week = daily[(daily.index >= d - pd.Timedelta(days=7)) & (daily.index < d)]
    tide_range_before[p] = week.range_ft.mean()

print(tide_range_before[:5])
missing = np.isnan(tide_range_before)
print(dates[missing])

[31.05357143 31.97642857 27.69028571 23.36142857 26.50071429]
DatetimeIndex(['2026-10-08', '2026-10-12', '2026-10-15'], dtype='datetime64[s]', freq=None)


In [ ]:
missing = np.isnan(tide_range_before)

params = {'station': '9455920', 'product': 'predictions', 'interval': 'hilo', 'datum': 'MLLW',
          'units': 'english', 'time_zone': 'gmt', 'format': 'csv',
          'begin_date': '20260901', 'end_date': '20261031'}
text = requests.get(TIDE_URL, params=params).text
pred = pd.read_csv(io.StringIO(text))
pred.columns = pred.columns.str.strip()
pred['time'] = pd.to_datetime(pred['Date Time'])

pday = pred.time.dt.floor('D')
pred_daily = pred.groupby(pday).Prediction.agg(['max', 'min'])
pred_range = pred_daily['max'] - pred_daily['min']

for p in np.where(missing)[0]:
    d = dates[p].normalize()
    week = pred_range[(pred_range.index >= d - pd.Timedelta(days=7)) & (pred_range.index < d)]
    tide_range_before[p] = week.mean()

print(dates[missing])
print(tide_range_before[missing])
print('missing now:', np.isnan(tide_range_before).sum())

DatetimeIndex(['2026-10-08', '2026-10-12', '2026-10-15'], dtype='datetime64[s]', freq=None)
[25.88642857 27.84857143 31.183     ]
missing now: 0


In [ ]:
tide_range_before[0]

np.float64(31.053571428571427)

In [ ]:
tide_daily = daily.range_ft.combine_first(pred_range)
tide_is_pred = ~tide_daily.index.isin(daily.index)

tide_daily = daily.range_ft.combine_first(pred_range)       # measured, + predictions for Oct 2026
tide_is_pred = ~tide_daily.index.isin(daily.index)         # True = predicted day

def mean_tide(start, end):
    """Average daily tidal range over the days start ... end-1."""
    return tide_daily[(tide_daily.index >= start) & (tide_daily.index < end)].mean()

days0 = dates.normalize()
tide_range_ahead = np.full(len(windows), np.nan)
for n, w in enumerate(windows):
    tide_range_ahead[n] = mean_tide(days0[w[3]], days0[w[4]])   # last input day ... day before target

print(len(tide_daily), 'days,', tide_is_pred.sum(), 'predicted')
print('ahead:', tide_range_ahead.shape, 'missing', np.isnan(tide_range_ahead).sum())
print(tide_range_ahead[:5])


4595 days, 31 predicted
ahead: (515,) missing 0
[29.85389286 29.58439286 29.53028571 29.76257143 30.08107143]


In [ ]:
import os, shutil

V2_DIR = '/content/drive/MyDrive/ehydro/anchorage/timeseries_v2'
os.makedirs(V2_DIR, exist_ok=True)

np.savez_compressed(f'{V2_DIR}/timeseries.npz',
    **{name: ts[name] for name in ts.files},          # everything from v1, unchanged
    tide_range_before=tide_range_before,               # (697,)  week before each map
    tide_range_ahead=tide_range_ahead,                 # (515,)  last input -> target
    tide_days=tide_daily.index.values,                 # (n_days,) dates
    tide_range_daily=tide_daily.values,                # (n_days,) range on each date
    tide_is_pred=tide_is_pred)                         # (n_days,) True = predicted

for f in ['timeseries_surveys.csv', 'dredged_cells.parquet']:
    shutil.copy(f'{TS_DIR}/{f}', V2_DIR)

v2 = np.load(f'{V2_DIR}/timeseries.npz')
for name in v2.files:
    print(name, v2[name].shape)

maps (697, 173, 91)
dredged (697, 173, 91)
core (173, 91)
windows (515, 5)
masks (515, 173, 91)
split (515,)
K ()
STEP ()
HORIZON ()
survey_id (697,)
dates (697,)
row0 ()
col0 ()
cell ()
tide_range_before (697,)
tide_range_ahead (515,)
tide_days (4595,)
tide_range_daily (4595,)
tide_is_pred (4595,)


River Flow:

Get the data for all three rivers. We save a table with data for each calendar day. This way its easy to change the range later. I might look into training with a longer date range since mud probably takes a while to accumulate.

In [5]:
USGS_URL = 'https://waterservices.usgs.gov/nwis/dv/'
RIVERS = {'knik': '15281000', 'matanuska': '15284000', 'shipcreek': '15276000'}

flows = {}
for name, site in RIVERS.items():
    params = {'format': 'rdb', 'sites': site, 'parameterCd': '00060',
              'startDT': '2014-01-01', 'endDT': '2026-11-03'}
    text = requests.get(USGS_URL, params=params).text
    table = pd.read_csv(io.StringIO(text), sep='\t', comment='#')
    table = table.iloc[1:]                                   # 2nd line is a format line, not data
    value_col = [c for c in table.columns if c.endswith('_00060_00003')][0]
    flow = pd.to_numeric(table[value_col], errors='coerce')  # text like 'Ice' -> NaN
    flow.index = pd.to_datetime(table['datetime'])
    flows[name] = flow

days = pd.date_range('2014-01-01', '2026-11-03', freq='D')   # every calendar day
river = pd.DataFrame(flows).reindex(days)
print(river.head())

print(river.shape)
print(river.notna().sum())
river.loc['2016-07-01':'2016-07-03']

            knik  matanuska  shipcreek
2014-01-01   NaN      900.0       55.1
2014-01-02   NaN      900.0       55.0
2014-01-03   NaN      900.0       55.0
2014-01-04   NaN      900.0       60.0
2014-01-05   NaN      850.0       60.0
(4690, 3)
knik         2471
matanuska    4628
shipcreek    4687
dtype: int64


,knik,matanuska,shipcreek
2016-07-01,28800.0,23900.0,258.0
2016-07-02,30000.0,24500.0,234.0
2016-07-03,30000.0,24700.0,221.0


Get the flow for the 7 days before each sample

In [6]:
def week_before(series, how):
    """For each map: mean or sum of a daily series over the 7 days before its survey."""
    out = np.full(len(dates), np.nan)
    for p, d in enumerate(dates.normalize()):
        week = series[(series.index >= d - pd.Timedelta(days=7)) & (series.index < d)]
        out[p] = week.mean() if how == 'mean' else week.sum(min_count=1)
    return out

flow_before = {name: week_before(river[name], 'mean') for name in RIVERS}

for name, arr in flow_before.items():
    print(name, arr.shape, 'missing', np.isnan(arr).sum())

knik (697,) missing 17
matanuska (697,) missing 0
shipcreek (697,) missing 0


In [10]:
V2_DIR = '/content/drive/MyDrive/ehydro/anchorage/timeseries_v2'
v2_path = f'{V2_DIR}/timeseries.npz'
old = np.load(v2_path)
arrays = {name: old[name] for name in old.files}     # everything already in v2 (incl. tides)
old.close()

arrays['days'] = days.values                         # shared calendar for rivers + weather
for name in RIVERS:
    arrays[f'{name}_flow_daily'] = river[name].values    # (n_days,)  cubic ft/s, NaN = no data
    arrays[f'{name}_flow_before'] = flow_before[name]    # (697,)     mean over 7 days before each map

np.savez_compressed(v2_path, **arrays)

v2 = np.load(v2_path)
for name in v2.files:
    print(name, v2[name].shape)

maps (697, 173, 91)
dredged (697, 173, 91)
core (173, 91)
windows (515, 5)
masks (515, 173, 91)
split (515,)
K ()
STEP ()
HORIZON ()
survey_id (697,)
dates (697,)
row0 ()
col0 ()
cell ()
tide_range_before (697,)
tide_range_ahead (515,)
tide_days (4595,)
tide_range_daily (4595,)
tide_is_pred (4595,)
days (4690,)
knik_flow_daily (4690,)
knik_flow_before (697,)
matanuska_flow_daily (4690,)
matanuska_flow_before (697,)
shipcreek_flow_daily (4690,)
shipcreek_flow_before (697,)


Weather. We save rain, snow and temp. We reuse the days map. The idea is that for river flow and weather everything should match up for any given day.

Download the data:

In [12]:
WEATHER_URL = 'https://archive-api.open-meteo.com/v1/archive'
params = {'latitude': 61.2375, 'longitude': -149.8904,          # the tide gauge, at the harbor
          'start_date': '2014-01-01', 'end_date': '2026-11-03',
          'daily': 'rain_sum,snowfall_sum,temperature_2m_mean',
          'timezone': 'America/Anchorage',
          'models': 'era5'}
data = requests.get(WEATHER_URL, params=params).json()

weather = pd.DataFrame(data['daily'])
weather['time'] = pd.to_datetime(weather['time'])
weather = weather.set_index('time').reindex(days)              # same calendar as the rivers
#print(weather.head())
weather.columns = ['rain', 'snowfall', 'temp']

print(data['daily_units'])
print(weather.shape)
print(weather.notna().sum())
weather.loc['2016-07-01':'2016-07-03']

            rain_sum  snowfall_sum  temperature_2m_mean
2014-01-01       5.1          0.35                  0.3
2014-01-02       1.8          0.00                  0.7
2014-01-03       0.6          0.00                  0.1
2014-01-04      10.3          0.00                  0.9
2014-01-05       2.2          0.00                  3.0
{'time': 'iso8601', 'rain_sum': 'mm', 'snowfall_sum': 'cm', 'temperature_2m_mean': '°C'}
(4690, 3)
rain        4686
snowfall    4686
temp        4686
dtype: int64


,rain,snowfall,temp
2016-07-01,6.6,0.0,14.7
2016-07-02,0.0,0.0,15.4
2016-07-03,0.0,0.0,16.6


Get 7 days before each sample. Reusing week_before form above.

In [13]:
weather_before = {
    'rain':     week_before(weather['rain'], 'sum'),       # total mm in the week
    'snowfall': week_before(weather['snowfall'], 'sum'),   # total cm in the week
    'temp':     week_before(weather['temp'], 'mean'),      # average °C
}

for name, arr in weather_before.items():
    print(name, arr.shape, 'missing', np.isnan(arr).sum(),
          'min', np.nanmin(arr).round(1), 'max', np.nanmax(arr).round(1))

rain (697,) missing 0 min 0.0 max 92.1
snowfall (697,) missing 0 min 0.0 max 3.3
temp (697,) missing 0 min -1.2 max 22.1


In [14]:
old = np.load(v2_path)
arrays = {name: old[name] for name in old.files}
old.close()

for name in ['rain', 'snowfall', 'temp']:
    arrays[f'{name}_daily'] = weather[name].values        # (n_days,) on the shared `days` calendar
    arrays[f'{name}_before'] = weather_before[name]       # (697,)

np.savez_compressed(v2_path, **arrays)

v2 = np.load(v2_path)
for name in v2.files:
    print(name, v2[name].shape)

maps (697, 173, 91)
dredged (697, 173, 91)
core (173, 91)
windows (515, 5)
masks (515, 173, 91)
split (515,)
K ()
STEP ()
HORIZON ()
survey_id (697,)
dates (697,)
row0 ()
col0 ()
cell ()
tide_range_before (697,)
tide_range_ahead (515,)
tide_days (4595,)
tide_range_daily (4595,)
tide_is_pred (4595,)
days (4690,)
knik_flow_daily (4690,)
knik_flow_before (697,)
matanuska_flow_daily (4690,)
matanuska_flow_before (697,)
shipcreek_flow_daily (4690,)
shipcreek_flow_before (697,)
rain_daily (4690,)
rain_before (697,)
snowfall_daily (4690,)
snowfall_before (697,)
temp_daily (4690,)
temp_before (697,)


In [15]:
for name in ['rain', 'snowfall', 'temp']:
    print(name, np.nanmin(arrays[f'{name}_before']).round(1), np.nanmax(arrays[f'{name}_before']).round(1))

rain 0.0 92.1
snowfall 0.0 3.3
temp -1.2 22.1


In [16]:
for name in ['rain', 'temp']:
    p = np.nanargmax(arrays[f'{name}_before'])
    print(name, dates[p].date(), arrays[f'{name}_before'][p].round(1))

rain 2025-09-15 92.1
temp 2019-07-11 22.1


In [17]:
print(arrays['dates'].shape)

(697,)
